In [39]:
import pandas as pd
import requests
from io import BytesIO
import numpy as np
import kagglehub
import os
import re
from thefuzz import fuzz, process

In [ ]:
realtor = pd.read_csv("https://econdata.s3-us-west-2.amazonaws.com/Reports/Core/RDC_Inventory_Core_Metrics_County_History.csv")
realtor_hotness = pd.read_csv("https://econdata.s3-us-west-2.amazonaws.com/Reports/Hotness/RDC_Inventory_Hotness_Metrics_County_History.csv")
realtor_merge = pd.merge(realtor, realtor_hotness, how="outer", on=["month_date_yyyymm", "county_fips"])

investor_data  = pd.read_csv("https://media.githubusercontent.com/media/walde363/covid-housing-forecast/main/data/raw/dowload_investor_purchases_by_metro.csv", encoding="utf-16", sep="\t")

BLS_unemployment = pd.read_csv("https://media.githubusercontent.com/media/walde363/covid-housing-forecast/main/data/raw/BLS_unemployment_data.csv")
BLS_weekly_earnings = pd.read_csv("https://media.githubusercontent.com/media/walde363/covid-housing-forecast/main/data/raw/BLS_weekly_earnings.csv")

## Open https://www.aoml.noaa.gov/hrd/hurdat/All_U.S._Hurricanes.html 
## Copy and paste table to a google sheet
## Download sheet as a csv
hurricanes_df = pd.read_csv("https://media.githubusercontent.com/media/walde363/covid-housing-forecast/main/data/raw/Huricanes.csv")

url = "https://www.freddiemac.com/pmms/docs/historicalweeklydata.xlsx"
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
response = requests.get(url, headers=headers)
response.raise_for_status()
mortgage_rates = pd.read_excel(BytesIO(response.content))

In [ ]:
# remove rows that contain s. This is a row to separate years
hurricanes_df_clean = hurricanes_df.copy(deep=True)
hurricanes_df_clean = hurricanes_df_clean[~hurricanes_df_clean["ar"].str.contains(r"[A-Za-z]", na=False)]

# flag rows with FL
hurricanes_df_clean["fl_flag"] = hurricanes_df_clean["States Affected and Category by States"].str.contains("FL", na=False)

hurricanes_df_clean = hurricanes_df_clean.rename(columns={
    'ar': 'year',
    'Highest\nSaffir-\nSimpson\nU.S. Category': 'highest_category'
})

#filter to FL
hurricanes_df_clean = hurricanes_df_clean[hurricanes_df_clean['fl_flag'] == True]

month_map = {
    "Jan": 1, "Feb": 2, "Mar": 3,
    "Apr": 4, "May": 5, "Jun": 6,
    "Jul": 7, "Aug": 8, "Sep": 9,
    "Oct": 10, "Nov": 11, "Dec": 12
}

# Make this cell safe to re-run: only expand when the raw Month column exists.
if "Month" in hurricanes_df_clean.columns:
    def expand_month(row):
        month_value = row["Month"]

        if month_value == "Sp-Oc":
            months = [9, 10]
        elif month_value == "Jl-Au":
            months = [7, 8]
        else:
            months = [month_map[month_value]]

        return pd.DataFrame({
            "year": row["year"],
            "month": months,
            "highest_category": pd.to_numeric(row["highest_category"], errors="coerce")
        })

    hurricanes_df_clean = pd.concat(
        hurricanes_df_clean.apply(expand_month, axis=1).tolist(),
        ignore_index=True
    )
elif {"year", "month", "highest_category"}.issubset(hurricanes_df_clean.columns):
    hurricanes_df_clean = hurricanes_df_clean.copy()
else:
    raise KeyError(
        "Expected either raw hurricane columns including 'Month' or transformed columns: "
        "'year', 'month', 'highest_category'."
    )

In [ ]:
investor_df_clean = investor_data.copy(deep=True)

# Split year and quarter
investor_df_clean[["year", "q"]] = investor_df_clean["Quarter"].str.split(" ", expand=True)

investor_df_clean["year"] = investor_df_clean["year"].astype(int)
investor_df_clean["quarter"] = investor_df_clean["q"].str.replace("Q", "").astype(int)

# Convert quarter → starting month
investor_df_clean["month"] = (investor_df_clean["quarter"] - 1) * 3 + 1

# Create datetime (quarter start)
investor_df_clean["date"] = pd.to_datetime(
    dict(year=investor_df_clean["year"], month=investor_df_clean["month"], day=1)
)

investor_df_clean["Investor Purchases"] = (
    investor_df_clean["Investor Purchases"]
    .str.replace(",", "", regex=False)
    .astype("Int32")
)

investor_df_clean["Investor Market Share"] = (
    investor_df_clean["Investor Market Share"]
    .str.replace("%", "", regex=False)
    .astype("Int32")
)

investor_df_clean["Investor Purchases YoY"] = (
    investor_df_clean["Investor Purchases YoY"]
    .str.replace("%", "", regex=False)
    .astype("Int32")
)

investor_df_clean = investor_df_clean.groupby(["year", "month", "date", "Redfin Metro"]).agg(
    {"Investor Purchases": "sum"
    ,"Investor Market Share": "mean"
    ,"Investor Purchases YoY": "mean"}).reset_index()

investor_df_clean[['city', 'state']] = investor_df_clean['Redfin Metro'].str.split(',', expand=True)
investor_df_clean = investor_df_clean.groupby(["date","state"]).agg({"Investor Purchases": "sum", "Investor Market Share": "mean", "Investor Purchases YoY": "mean"}).reset_index()
investor_df_clean['state'] = investor_df_clean['state'].str.lower().str.strip()

In [ ]:
mortgage_rates_df_clean = mortgage_rates.copy(deep=True)
mortgage_rates_df_clean = mortgage_rates_df_clean[mortgage_rates_df_clean['Unnamed: 0'].notnull()]

# take first row as header
mortgage_rates_df_clean.columns = mortgage_rates_df_clean.iloc[0]
mortgage_rates_df_clean = mortgage_rates_df_clean[1:].reset_index(drop=True)

week_parsed = pd.to_datetime(mortgage_rates_df_clean["Week"], errors="coerce")
mortgage_rates_df_clean = mortgage_rates_df_clean[week_parsed.notna()].copy()
mortgage_rates_df_clean["Week"] = week_parsed[week_parsed.notna()].dt.date

# renaming columns from base spreedsheet
mortgage_rates_df_clean.columns.values[0] = "Week"
mortgage_rates_df_clean.columns.values[1] = "U.S. 30 year FRM"
mortgage_rates_df_clean.columns.values[2] = "30 year fees & points"

mortgage_rates_df_clean.columns.values[3] = "U.S. 15 year FRM"
mortgage_rates_df_clean.columns.values[4] = "15 year fees & points"

mortgage_rates_df_clean.columns.values[5] = "U.S. 5/1 ARM"
mortgage_rates_df_clean.columns.values[6] = "5/1 year fees & points"

mortgage_rates_df_clean.columns.values[7] = "U.S. 5/1 ARM margin"
mortgage_rates_df_clean.columns.values[8] = "30 year FRM / 5/1 ARM spread"

#Truncate
mortgage_rates_df_clean = mortgage_rates_df_clean[mortgage_rates_df_clean['Week']>=pd.to_datetime("2016-01-01").date()]

mortgage_rates_df_clean['Week'] = pd.to_datetime(mortgage_rates_df_clean['Week'])
mortgage_rates_df_clean['Month'] = mortgage_rates_df_clean['Week'].dt.to_period('M').dt.to_timestamp()
mortgage_rates_df_clean.drop(columns=['Week'], inplace=True)
mortgage_rates_df_clean = mortgage_rates_df_clean.groupby('Month').mean().reset_index()

In [ ]:
BLS_unemployment_data_df_clean = BLS_unemployment.copy(deep=True)

BLS_unemployment_data_df_clean = pd.melt(BLS_unemployment_data_df_clean,
                                 id_vars=["State","Area"],
                                 var_name="Month",
                                 value_name="Unemployment_Rate")

# statiscal area names were cleaned for join but there is not a 1 for 1 match. One example is below.
# Alachua is an example of a county not in earnings data
# BLS_weekly_earnings_df_clean[BLS_weekly_earnings_df_clean['State']=='Florida']['Area'].unique()

BLS_unemployment_data_df_clean['Area'] = BLS_unemployment_data_df_clean['Area'].str.replace(" Metropolitan Statistical Area", "", regex=False)

BLS_unemployment_data_df_clean = BLS_unemployment_data_df_clean[BLS_unemployment_data_df_clean['State'] != BLS_unemployment_data_df_clean['Area']]

In [ ]:
BLS_weekly_earnings_df_clean = BLS_weekly_earnings.copy(deep=True)

BLS_weekly_earnings_df_clean['Area'] = np.where(
    BLS_weekly_earnings_df_clean['Area'] == 'Statewide',
    BLS_weekly_earnings_df_clean['State'],
    BLS_weekly_earnings_df_clean['Area']
)

BLS_weekly_earnings_df_clean = pd.melt(BLS_weekly_earnings_df_clean,
                                 id_vars=["State","Area"],
                                 var_name="Month",
                                 value_name="Earnings")

BLS_weekly_earnings_df_clean = BLS_weekly_earnings_df_clean[BLS_weekly_earnings_df_clean['State'] != BLS_weekly_earnings_df_clean['Area']]
BLS_weekly_earnings_df_clean = BLS_weekly_earnings_df_clean[~BLS_weekly_earnings_df_clean['Area'].str.contains('Metropolitan Division', na=False)]

In [ ]:
BLS_df_clean_merge = pd.merge(BLS_weekly_earnings_df_clean,BLS_unemployment_data_df_clean,how="outer", on=["State","Area","Month"])
BLS_df_clean_merge[['city', 'state']] = BLS_df_clean_merge['Area'].str.split(', ', expand=True)
BLS_df_clean_merge['city'] = BLS_df_clean_merge['city'].str.lower()
BLS_df_clean_merge['state'] = BLS_df_clean_merge['state'].str.lower()
BLS_df_clean_merge['Month'] = pd.to_datetime(BLS_df_clean_merge['Month'].str.replace('M', '', regex=False).str.replace('-', '', regex=False), format='%Y%m')
BLS_df_clean_merge = BLS_df_clean_merge.groupby(["state","Month"]).agg({"Earnings": "mean", "Unemployment_Rate": "mean"}).reset_index()

In [ ]:
realtor_merge = pd.merge(realtor, realtor_hotness, how="outer", on=["month_date_yyyymm", "county_fips"])
realtor_merge[['city', 'state']] = realtor_merge['county_name_x'].str.split(', ', expand=True)
realtor_merge['month_date_yyyymm'] = pd.to_datetime(realtor_merge['month_date_yyyymm'], format='%Y%m')

In [ ]:
realtor_bls_merge = pd.merge(realtor_merge, BLS_df_clean_merge, how="left", left_on=["state", "month_date_yyyymm"], right_on=["state", "Month"])

In [ ]:
#by quarter by state - resample to monthly, dividing values by 3 to preserve scale
numeric_cols = ['Investor Purchases', 'Investor Market Share', 'Investor Purchases YoY']
investor_df_clean[numeric_cols] = investor_df_clean[numeric_cols] / 3
investor_df_clean = investor_df_clean.set_index('date').groupby('state').resample('MS').ffill().drop(columns='state').reset_index()

In [ ]:
hurricanes_df_clean['month_date_yyyymm'] = pd.to_datetime(hurricanes_df_clean['year'].astype(str) + '-' + hurricanes_df_clean['month'].astype(str).str.zfill(2) + '-01')

In [ ]:
processed_data_pre_model = realtor_bls_merge.copy(deep=True)
processed_data_pre_model = pd.merge(processed_data_pre_model, investor_df_clean, left_on=['month_date_yyyymm','state'], right_on=["date","state"], how="left")
processed_data_pre_model = pd.merge(processed_data_pre_model, mortgage_rates_df_clean, left_on="month_date_yyyymm", right_on="Month", how="left")
processed_data_pre_model = pd.merge(processed_data_pre_model, hurricanes_df_clean, left_on="month_date_yyyymm", right_on="month_date_yyyymm", how="left")

In [ ]:
processed_data_pre_model.drop(columns=(['date','year','Month_x','Month_y']), inplace=True)
processed_data_pre_model.rename(columns={"month_date_yyyymm": "date"}, inplace=True)

In [ ]:
processed_data_pre_model.to_csv("../data/processed/processed_data_pre_model.csv", index=False)